# Exact residue coverage before receptor preparation

Consumer evidence for DockingMT #5/#33 and MolSysMT #218. Exact inventory and
stored-graph comparison do not certify hydrogen completeness, protonation,
charge/type assignment, repair or docking readiness. See
`receptor_coverage_consumption.md` for the source contract and limitations.

Execute from this checkout with `molsyssuite@uibcdf_3.14` and the exact MolSysMT
source below on `PYTHONPATH`. No provider checkout or input file is edited.
The final cell writes new raw evidence under `data/receptor_coverage/`; historical
readiness-only measurements are preserved.

In [1]:
import hashlib
import importlib
import json
from pathlib import Path
from statistics import median
import sys
from time import perf_counter

import molsysmt as msm
import numpy as np
import pandas as pd
import pyunitwizard as puw
from dockingmt import prepare_receptor
from dockingmt._private.smonitor import ArgumentError
from dockingmt.preparation._molsys import chemistry_evidence

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'tests/data/vina_torsions/1iep_receptorH.pdb').is_file())
provider_source = 'e8e4fff22d0df0d26a3b91d80ea5a85c04981aef'
identity = {
    'python': sys.version.split()[0], 'interpreter': sys.executable,
    'provider_import': msm.__file__, 'qualified_provider_source': provider_source,
    'implementation_sha256': {
        name: hashlib.sha256((Path(msm.__file__).parent / name).read_bytes()).hexdigest()
        for name in ['build/get_residue_chemical_coverage.py', '_private/chemical_readiness.py']
    },
}
print(json.dumps(identity, indent=2))

{
  "python": "3.14.7",
  "interpreter": "/home/diego/Myopt/miniconda3/envs/molsyssuite@uibcdf_3.14/bin/python",
  "provider_import": "/tmp/dockingmt-receptor-provider-e8e4fff22d0df0d26a3b91d80ea5a85c04981aef/molsysmt/__init__.py",
  "qualified_provider_source": "e8e4fff22d0df0d26a3b91d80ea5a85c04981aef",
  "implementation_sha256": {
    "build/get_residue_chemical_coverage.py": "6afb3b001a3e7bb68f57c3f7904bc39e482ed1f6685b6f9e08416c459303f38c",
    "_private/chemical_readiness.py": "42a6b38e011c0a429a44d468172da230d89130ae143174b02a60b4ebdb9fd74b"
  }
}


## Unmodified real sources

181L includes protein, benzene and water groups. 1IEP is the original hydrogenated
PDB from the pinned Vina example, not its already prepared 2,702-atom PDBQT.
All counts below refer to the selected source before hydrogen projection.
Whole groups are audited; preparation extraction precedes the provider call.

In [2]:
paths = {
    '181L': Path(msm.systems['T4 lysozyme L99A']['181l.pdb']),
    '1IEP': root / 'tests/data/vina_torsions/1iep_receptorH.pdb',
}
assert hashlib.sha256(paths['1IEP'].read_bytes()).hexdigest() == '5f6aee6029f9a2a2c2be32d4eb948ae70808690573e1b69a0850cdffd7048ca7'
sources, observations, rows = {}, [], []
for name, path in paths.items():
    original = path.read_bytes()
    sources[name] = msm.convert(str(path), to_form='molsysmt.MolSys')
    for selection in ['all', "molecule_type=='protein'"]:
        selected = msm.extract(sources[name], selection=selection)
        evidence = chemistry_evidence(selected, include_residue_coverage=True)
        full = msm.build.get_residue_chemical_coverage(
            selected, chemical_state='structure', structure_indices=0)
        record = {'case': name, 'selection': selection,
                  'source_sha256': hashlib.sha256(original).hexdigest(),
                  'source_atoms': msm.get(selected, n_atoms=True),
                  'coverage': evidence['residue_coverage'],
                  'unassessed_group_name_counts': dict(pd.Series([
                      group['group_name'] for group in full['groups']
                      if group['status'] == 'unassessed']).value_counts().items()),
                  'hydrogen_reason_counts': dict(pd.Series([
                      group['hydrogens']['reason_code'] for group in full['groups']
                  ]).value_counts().items())}
        observations.append(record)
        rows.append({'case': name, 'selection': selection, 'atoms': record['source_atoms'],
                     'groups': evidence['residue_coverage']['n_groups'],
                     **evidence['residue_coverage']['status_counts'],
                     'H_unassessed': evidence['residue_coverage']['check_status_counts']['hydrogens'].get('unassessed', 0)})
    assert path.read_bytes() == original
display(pd.DataFrame(rows))
print(json.dumps([{key: value for key, value in record.items()
                   if key in ['case', 'selection', 'unassessed_group_name_counts', 'hydrogen_reason_counts']}
                  for record in observations], indent=2))

,case,selection,atoms,groups,assessed,incomplete,unassessed,H_unassessed
0,181L,all,1441,302,162,0,140,302
1,181L,molecule_type=='protein',1289,162,162,0,0,162
2,1IEP,all,4412,274,274,0,0,274
3,1IEP,molecule_type=='protein',4412,274,274,0,0,274


[
  {
    "case": "181L",
    "selection": "all",
    "unassessed_group_name_counts": {
      "HOH": 136,
      "CL": 2,
      "HED": 1,
      "BNZ": 1
    },
    "hydrogen_reason_counts": {
      "ambiguous_hydrogen_template_variants": 162,
      "no_exact_residue_template": 140
    }
  },
  {
    "case": "181L",
    "selection": "molecule_type=='protein'",
    "unassessed_group_name_counts": {},
    "hydrogen_reason_counts": {
      "ambiguous_hydrogen_template_variants": 162
    }
  },
  {
    "case": "1IEP",
    "selection": "all",
    "unassessed_group_name_counts": {},
    "hydrogen_reason_counts": {
      "ambiguous_hydrogen_template_variants": 274
    }
  },
  {
    "case": "1IEP",
    "selection": "molecule_type=='protein'",
    "unassessed_group_name_counts": {},
    "hydrogen_reason_counts": {
      "ambiguous_hydrogen_template_variants": 274
    }
  }
]


## Controlled missing, modified and unsupported chemistry

The ALA lacking CB is incomplete. MSE and SEP have their own exact heavy-atom
templates; MSE must retain Se rather than borrow MET sulfur. Unknown PTR, water
and zinc are explicitly unassessed. The controlled fixtures have no H atoms and
are software checks, not physically plausible coordinate models.

In [3]:
builder = msm.MolSysBuilder()
specifications = [
    ('ALA', ['N','CA','C','O','CB'], ['N','C','C','O','C']),
    ('ALA', ['N','CA','C','O'], ['N','C','C','O']),
    ('MSE', ['N','CA','C','O','CB','CG','SE','CE'], ['N','C','C','O','C','C','Se','C']),
    ('SEP', ['N','CA','C','O','CB','OG','P','O1P','O2P','O3P'], ['N','C','C','O','C','O','P','O','O','O']),
    ('PTR', ['N'], ['N']), ('HOH', ['O'], ['O']), ('ZN', ['ZN'], ['Zn']),
]
edges = [('N','CA'),('CA','C'),('C','O'),('CA','CB'),('CB','CG'),('CG','SE'),
         ('SE','CE'),('CB','OG'),('OG','P'),('P','O1P'),('P','O2P'),('P','O3P')]
for group_id, (name, names, elements) in enumerate(specifications, start=1):
    lookup = {name: builder.add_atom(atom_name=name, atom_type=element)
              for name, element in zip(names, elements)}
    builder.add_group(list(lookup.values()), group_name=name, group_id=group_id)
    for first, second in edges:
        if first in lookup and second in lookup:
            builder.add_bond(lookup[first], lookup[second],
                             bond_order=2 if second in ('O','O1P') else 1,
                             bond_type='covalent')
builder.set_coordinates(puw.quantity(np.zeros((1, builder.n_atoms, 3)), 'nm'))
control = builder.build()
before_states = msm.convert(control.chemical_states, to_form='molsysmt.ChemicalStatesDict').to_dict()
before_coordinates = puw.get_value(control.structures.coordinates, to_unit='nm').copy()
report = msm.build.get_residue_chemical_coverage(control, chemical_state='structure', structure_indices=0)
assert report['summary'] == {'assessed': 3, 'incomplete': 1, 'unassessed': 3}
diagnostics = [{
    'group': group['group_name'], 'status': group['status'],
    'reasons': group['reason_codes'],
    'missing_heavy': group['heavy_atoms']['missing_atom_names'],
    'H_status': group['hydrogens']['status'],
    'H_reason': group['hydrogens']['reason_code'],
    'template': group['template']['group_name'] if group['template'] else None,
} for group in report['groups']]
assert report['groups'][1]['heavy_atoms']['missing_atom_names'] == ['CB']
assert 'SE' in report['groups'][2]['heavy_atoms']['expected_atom_names']
assert 'SD' not in report['groups'][2]['heavy_atoms']['expected_atom_names']
display(pd.DataFrame(diagnostics))
np.testing.assert_equal(msm.convert(control.chemical_states, to_form='molsysmt.ChemicalStatesDict').to_dict(), before_states)
np.testing.assert_array_equal(puw.get_value(control.structures.coordinates, to_unit='nm'), before_coordinates)

,group,status,reasons,missing_heavy,H_status,H_reason,template
0,ALA,assessed,[],[],unassessed,ambiguous_hydrogen_template_variants,ALA
1,ALA,incomplete,[missing_heavy_atoms],[CB],unassessed,incomplete_or_conflicting_atom_mapping,ALA
2,MSE,assessed,[],[],unassessed,heavy_only_template,MSE
3,SEP,assessed,[],[],unassessed,heavy_only_template,SEP
4,PTR,unassessed,[no_exact_residue_template],None,unassessed,no_exact_residue_template,None
5,HOH,unassessed,[no_exact_residue_template],None,unassessed,no_exact_residue_template,None
6,ZN,unassessed,[no_exact_residue_template],None,unassessed,no_exact_residue_template,None


## Preparation remains provisional; original 1IEP still fails clearly

The new evidence adds no bonds, H atoms, charges or AutoDock assignments. The
default Vina guard is verified by the existing real-engine tests. Original 1IEP
has two hydrogens without the explicit heavy-atom attachment required by the
writer, despite assessed heavy-atom inventories. Its ambiguous H inventories
cannot establish preparation eligibility. No repair is performed here.

In [4]:
prepared = prepare_receptor(sources['181L'], selection="molecule_type=='protein'")
assert prepared.metadata['charge_source'] == 'zero_placeholder'
assert 'heuristic' in prepared.metadata['atom_type_source']
assert 'docking_readiness' in prepared.metadata['source_chemistry']['residue_coverage']['unassessed_checks']
print(json.dumps({'retained_atoms': prepared.n_atoms,
                  'charge_source': prepared.metadata['charge_source'],
                  'atom_type_source': prepared.metadata['atom_type_source'],
                  'residue_status': prepared.metadata['source_chemistry']['residue_coverage']['status_counts']}, indent=2))
try:
    prepare_receptor(sources['1IEP'], selection="molecule_type=='protein'")
except ArgumentError as error:
    assert 'exactly one explicit heavy-atom bond' in str(error)
    print('Original 1IEP, without repair:', str(error))
else:
    raise AssertionError('The original unsupported preparation case changed; review its evidence.')

{
  "retained_atoms": 1289,
  "charge_source": "zero_placeholder",
  "atom_type_source": "element_residue_heuristic",
  "residue_status": {
    "assessed": 162,
    "incomplete": 0,
    "unassessed": 0
  }
}
Original 1IEP, without repair: Invalid argument 'molecular_system': A receptor hydrogen needs exactly one explicit heavy-atom bond for Vina preparation.


## Paired local cost, with unchanged PDBQT bytes

Five paired warm-cache rounds alternate order after one warm-up per profile.
The baseline uses the current stored-field-only summary, while the coverage
profile uses the new combined provider report, reusing its stored-field audit.
For 181L, measure complete receptor preparation with the same source/selection;
for 1IEP measure source assessment only because unmodified preparation fails.
These are different operations. Templates are already warm from earlier cells.
Time excludes conversion from the original PDB. OS caches/load are uncontrolled;
these measurements establish neither a general time bound nor peak memory.
JSON byte counts measure serialized metadata only.

In [5]:
module = importlib.import_module('dockingmt.preparation.receptor')
current = module.chemistry_evidence
def readiness_only(molsys, **_kwargs):
    return current(molsys)

profiles = [('stored_fields', readiness_only), ('residue_coverage', current)]
measurements = []
try:
    for case, operation in [('181L', 'receptor_preparation'), ('1IEP', 'source_assessment')]:
        samples = {profile: [] for profile, _ in profiles}
        results = {}
        def execute(evidence):
            if operation == 'receptor_preparation':
                module.chemistry_evidence = evidence
                return prepare_receptor(sources[case], selection="molecule_type=='protein'")
            return evidence(sources[case], include_residue_coverage=True)
        for _, evidence in profiles:
            execute(evidence)
        for round_index in range(5):
            for profile, evidence in (profiles if round_index % 2 == 0 else profiles[::-1]):
                start = perf_counter()
                results[profile] = execute(evidence)
                samples[profile].append(perf_counter() - start)
        if operation == 'receptor_preparation':
            assert results['stored_fields'].to_pdbqt() == results['residue_coverage'].to_pdbqt()
        metadata = {profile: result.metadata if operation == 'receptor_preparation' else result
                    for profile, result in results.items()}
        fields = {profile: value['source_chemistry']['chemical_readiness']
                  if operation == 'receptor_preparation' else value['chemical_readiness']
                  for profile, value in metadata.items()}
        assert fields['stored_fields'] == fields['residue_coverage']
        for profile, _ in profiles:
            measurements.append({'case': case, 'operation': operation, 'profile': profile,
                                 'samples_seconds': samples[profile],
                                 'median_seconds': median(samples[profile]),
                                 'metadata_bytes': len(json.dumps(metadata[profile], sort_keys=True).encode())})
finally:
    module.chemistry_evidence = current

destination = root / 'devguide/validation/data/receptor_coverage'
destination.mkdir(parents=True, exist_ok=True)
(destination / 'cases.json').write_text(json.dumps({'identity': identity, 'cases': observations,
                                                   'controls': diagnostics}, indent=2) + '\n')
(destination / 'profile.json').write_text(json.dumps({
    'identity': identity, 'baseline': 'Current preparation with stored-field evidence only',
    'sampling': 'Five paired rounds alternating order after one warm-up per profile; warm templates',
    'evidence': 'Local source-specific timings and JSON sizes, not universal speed/memory claims',
    'pdbqt_equivalence': {'181L': True}, 'cases': measurements}, indent=2) + '\n')
display(pd.DataFrame([{key: value for key, value in row.items() if key != 'samples_seconds'}
                      for row in measurements]))

,case,operation,profile,median_seconds,metadata_bytes
0,181L,receptor_preparation,stored_fields,0.168151,10773
1,181L,receptor_preparation,residue_coverage,0.365577,14779
2,1IEP,source_assessment,stored_fields,0.033735,3810
3,1IEP,source_assessment,residue_coverage,0.390975,7993
